# Hermes Agent on Google Colab

Google Colab の A100 GPU 上で、Ollama の `gemma4:31b` を使って Hermes Agent を動かします。
Hermes の記憶・設定・SQLite・skills と作業ファイルは、各ターン後に Google Drive へ同期します。

**対象:** Colab Pro / Pro+ 等で A100 80 GB 以上を利用できる方

**前提:**

- ランタイムのハードウェア アクセラレータを **A100 GPU** に設定済みであること
- Google Drive に少なくとも Agent 状態と作業ファイルを保存できる空きがあること
- モデル約 20 GB はランタイムごとに取得するため、十分なランタイムディスクと通信量があること

**重要:** セルは上から順に実行してください。`allow_dangerous=True` は、その呼び出しに限って
Hermes の危険操作承認を省略します。信頼できる指示だけに使用してください。


## 実行手順

1. 定数と保存先を初期化する
2. Google Drive をマウントし、A100・RAM・ディスクを検査する
3. 永続状態を Drive からローカルへ復元する
4. Hermes Agent と Ollama を導入・設定する
5. `gemma4:31b` を取得して疎通確認する
6. `ask_hermes()` または `%llm` で会話し、各ターン後に自動保存する
7. 必要に応じてWeb DashboardまたはDesktop App接続を起動する


## 1. 定数とローカル保存先

SQLite は Google Drive の FUSE 上ではなく `/content` で動かします。
DriveへはSQLite Backup APIで作成した正常なコピーだけを保存します。


In [ ]:
from __future__ import annotations

import atexit
import importlib.metadata
import json
import os
import re
import shutil
import signal
import sqlite3
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

MODEL = "gemma4:31b"
HERMES_VERSION = "0.17.0"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
OLLAMA_CONTEXT_LENGTH = "65536"

LOCAL_HOME = Path("/content/hermes-home")
LOCAL_WORKSPACE = Path("/content/hermes-workspace")
ARTIFACTS_DIR = LOCAL_WORKSPACE / "artifacts"
LOCAL_OLLAMA_MODELS = Path("/content/ollama-models")
OLLAMA_LOG = Path("/content/ollama.log")

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / "Hermes"
DRIVE_STATE = DRIVE_ROOT / "state"
DRIVE_WORKSPACE = DRIVE_ROOT / "workspace"
DRIVE_SNAPSHOTS = DRIVE_ROOT / "snapshots"

RUNTIME_STATE_FILE = LOCAL_HOME / "colab_runtime.json"
CONFIG_FILE = LOCAL_HOME / "config.yaml"

# Hermes must see this before any Hermes module or subprocess starts.
os.environ["HERMES_HOME"] = str(LOCAL_HOME)
os.environ["TERMINAL_CWD"] = str(LOCAL_WORKSPACE)
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_CONTEXT_LENGTH"] = OLLAMA_CONTEXT_LENGTH
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_MODELS"] = str(LOCAL_OLLAMA_MODELS)
os.environ["HERMES_ARTIFACTS_DIR"] = str(ARTIFACTS_DIR)

for directory in (LOCAL_HOME, LOCAL_WORKSPACE, ARTIFACTS_DIR, LOCAL_OLLAMA_MODELS):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Hermes home: {LOCAL_HOME}")
print(f"Workspace:   {LOCAL_WORKSPACE}")
print(f"Artifacts:   {ARTIFACTS_DIR}")
print(f"Drive root:  {DRIVE_ROOT}")


## 2. Driveマウントとハードウェア検査

条件不足の場合はモデルを勝手に変更せず、ここで停止します。A100の割り当てはColab側で変更してください。


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("このNotebookはGoogle Colab上で実行してください。") from exc

drive.mount(str(DRIVE_MOUNT))

def _run_text(command: list[str]) -> str:
    return subprocess.check_output(command, text=True, stderr=subprocess.STDOUT).strip()

try:
    gpu_line = _run_text([
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader,nounits",
    ]).splitlines()[0]
    gpu_name, gpu_memory_text = [part.strip() for part in gpu_line.rsplit(",", 1)]
    gpu_memory_mib = int(gpu_memory_text)
except (FileNotFoundError, subprocess.CalledProcessError, ValueError, IndexError) as exc:
    raise RuntimeError("NVIDIA GPUを検出できません。ColabのランタイムをA100 GPUへ変更してください。") from exc

ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_free_gib = shutil.disk_usage("/content").free / 1024**3

failures = []
if "A100" not in gpu_name.upper() or gpu_memory_mib < 39_000:
    failures.append(f"GPU: {gpu_name} ({gpu_memory_mib / 1024:.1f} GiB VRAM)")
if ram_gib < 24:
    failures.append(f"RAM: {ram_gib:.1f} GiB（24 GiB以上が必要）")
if disk_free_gib < 30:
    failures.append(f"空きディスク: {disk_free_gib:.1f} GiB（30 GiB以上が必要）")
if failures:
    raise RuntimeError("実行要件を満たしていません:\n- " + "\n- ".join(failures))

for directory in (DRIVE_ROOT, DRIVE_STATE, DRIVE_WORKSPACE, DRIVE_SNAPSHOTS):
    directory.mkdir(parents=True, exist_ok=True)

print(f"GPU:       {gpu_name} ({gpu_memory_mib / 1024:.1f} GiB VRAM)")
print(f"RAM:       {ram_gib:.1f} GiB")
print(f"Disk free: {disk_free_gib:.1f} GiB")
print("Hardware preflight: OK")


## 3. 復元・チェックポイント機能

`checkpoint()` はAgent状態を一旦ローカルに構築し、全SQLiteを検査してからDriveへ反映します。
`restore()` は現在状態が壊れていれば、新しい順に正常なスナップショットを探します。


In [ ]:
HOME_EXCLUDED_DIRS = {
    ".git", "__pycache__", "audio_cache", "backups", "bin", "cache",
    "hermes-agent", "image_cache", "logs", "node", "ollama",
}
HOME_EXCLUDED_FILES = {".env"}
WORKSPACE_EXCLUDED_DIRS = {".ipynb_checkpoints", "__pycache__"}

def _is_relative_to(path: Path, parent: Path) -> bool:
    try:
        path.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False

def _assert_drive_target(path: Path) -> None:
    if not _is_relative_to(path, DRIVE_ROOT):
        raise ValueError(f"Drive操作対象がHermesディレクトリ外です: {path}")

def _sqlite_integrity(path: Path) -> tuple[bool, str]:
    try:
        connection = sqlite3.connect(f"file:{path}?mode=ro", uri=True, timeout=10)
        try:
            row = connection.execute("PRAGMA integrity_check").fetchone()
        finally:
            connection.close()
        result = str(row[0]) if row else "no result"
        return result.lower() == "ok", result
    except sqlite3.Error as exc:
        return False, str(exc)

def _backup_sqlite(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    src = sqlite3.connect(str(source), timeout=30)
    dst = sqlite3.connect(str(destination), timeout=30)
    try:
        src.backup(dst)
    finally:
        dst.close()
        src.close()
    ok, detail = _sqlite_integrity(destination)
    if not ok:
        destination.unlink(missing_ok=True)
        raise RuntimeError(f"SQLiteバックアップの整合性検査に失敗しました: {source}: {detail}")

def _excluded(relative: Path, excluded_dirs: set[str], excluded_files: set[str]) -> bool:
    if any(part in excluded_dirs for part in relative.parts):
        return True
    if relative.name in excluded_files:
        return True
    return relative.name.endswith(("-wal", "-shm"))

def _copy_home_for_checkpoint(source_root: Path, destination_root: Path) -> None:
    destination_root.mkdir(parents=True, exist_ok=True)
    for source_path in source_root.rglob("*"):
        relative = source_path.relative_to(source_root)
        if _excluded(relative, HOME_EXCLUDED_DIRS, HOME_EXCLUDED_FILES):
            continue
        if source_path.is_symlink():
            continue
        destination_path = destination_root / relative
        if source_path.is_dir():
            destination_path.mkdir(parents=True, exist_ok=True)
        elif source_path.suffix == ".db":
            _backup_sqlite(source_path, destination_path)
        elif source_path.is_file():
            destination_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination_path)

def _validate_state_tree(root: Path) -> tuple[bool, str]:
    for database in root.rglob("*.db"):
        ok, detail = _sqlite_integrity(database)
        if not ok:
            return False, f"{database}: {detail}"
    return True, "ok"

def _replace_drive_directory(source: Path, destination: Path) -> None:
    _assert_drive_target(destination)
    token = uuid.uuid4().hex[:10]
    pending = DRIVE_ROOT / f".{destination.name}-next-{token}"
    previous = DRIVE_ROOT / f".{destination.name}-previous-{token}"
    _assert_drive_target(pending)
    _assert_drive_target(previous)
    shutil.rmtree(pending, ignore_errors=True)
    shutil.copytree(source, pending)
    if destination.exists():
        destination.rename(previous)
    pending.rename(destination)
    shutil.rmtree(previous, ignore_errors=True)

def _copy_tree(source: Path, destination: Path, excluded_dirs: set[str] | None = None) -> None:
    excluded_dirs = excluded_dirs or set()
    destination.mkdir(parents=True, exist_ok=True)
    for source_path in source.rglob("*"):
        relative = source_path.relative_to(source)
        if any(part in excluded_dirs for part in relative.parts) or source_path.is_symlink():
            continue
        destination_path = destination / relative
        if source_path.is_dir():
            destination_path.mkdir(parents=True, exist_ok=True)
        elif source_path.is_file():
            destination_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination_path)

def _snapshot_candidates() -> list[Path]:
    if not DRIVE_SNAPSHOTS.exists():
        return []
    pattern = re.compile(r"^\d{8}T\d{6}Z-[0-9a-f]{6}$")
    return sorted(
        (path for path in DRIVE_SNAPSHOTS.iterdir() if path.is_dir() and pattern.match(path.name)),
        reverse=True,
    )

def _prune_snapshots(keep: int = 3) -> None:
    for old_snapshot in _snapshot_candidates()[keep:]:
        _assert_drive_target(old_snapshot)
        shutil.rmtree(old_snapshot)

def checkpoint() -> Path:
    """Persist Hermes state and workspace to Drive; keep three valid state snapshots."""
    if not DRIVE_MOUNT.exists() or not DRIVE_ROOT.exists():
        raise RuntimeError("Google Driveがマウントされていません。")
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    snapshot_name = f"{timestamp}-{uuid.uuid4().hex[:6]}"
    with tempfile.TemporaryDirectory(prefix="hermes-checkpoint-", dir="/content") as temp_dir:
        local_snapshot = Path(temp_dir) / "state"
        _copy_home_for_checkpoint(LOCAL_HOME, local_snapshot)
        ok, detail = _validate_state_tree(local_snapshot)
        if not ok:
            raise RuntimeError(f"チェックポイントを中止しました: {detail}")

        drive_pending = DRIVE_SNAPSHOTS / f".{snapshot_name}.pending"
        drive_snapshot = DRIVE_SNAPSHOTS / snapshot_name
        _assert_drive_target(drive_pending)
        shutil.rmtree(drive_pending, ignore_errors=True)
        shutil.copytree(local_snapshot, drive_pending)
        ok, detail = _validate_state_tree(drive_pending)
        if not ok:
            shutil.rmtree(drive_pending, ignore_errors=True)
            raise RuntimeError(f"Driveへの書き込み検査に失敗しました: {detail}")
        drive_pending.rename(drive_snapshot)
        _replace_drive_directory(local_snapshot, DRIVE_STATE)

    with tempfile.TemporaryDirectory(prefix="hermes-workspace-", dir="/content") as temp_dir:
        workspace_snapshot = Path(temp_dir) / "workspace"
        _copy_tree(LOCAL_WORKSPACE, workspace_snapshot, WORKSPACE_EXCLUDED_DIRS)
        _replace_drive_directory(workspace_snapshot, DRIVE_WORKSPACE)

    _prune_snapshots(keep=3)
    print(f"Checkpoint saved: {drive_snapshot.name}")
    return drive_snapshot

def _select_restore_state() -> Path | None:
    candidates = [DRIVE_STATE, *_snapshot_candidates()]
    for candidate in candidates:
        if not candidate.exists() or not any(candidate.iterdir()):
            continue
        ok, detail = _validate_state_tree(candidate)
        if ok:
            return candidate
        print(f"Skipping invalid state {candidate.name}: {detail}")
    return None

def restore(force: bool = False) -> Path | None:
    """Restore the newest valid state and workspace. Use force=True to replace local data."""
    if not DRIVE_ROOT.exists():
        raise RuntimeError("Google DriveのHermesディレクトリが見つかりません。")
    state_source = _select_restore_state()
    local_has_state = any(LOCAL_HOME.iterdir())
    if state_source is not None and (force or not local_has_state):
        if force:
            shutil.rmtree(LOCAL_HOME)
            LOCAL_HOME.mkdir(parents=True, exist_ok=True)
        _copy_tree(state_source, LOCAL_HOME)
        print(f"State restored from: {state_source}")
    elif state_source is None:
        print("No saved Hermes state found; starting fresh.")
    else:
        print("Local Hermes state already exists; restore skipped.")

    local_has_workspace = any(LOCAL_WORKSPACE.iterdir())
    if DRIVE_WORKSPACE.exists() and any(DRIVE_WORKSPACE.iterdir()) and (force or not local_has_workspace):
        if force:
            shutil.rmtree(LOCAL_WORKSPACE)
            LOCAL_WORKSPACE.mkdir(parents=True, exist_ok=True)
        _copy_tree(DRIVE_WORKSPACE, LOCAL_WORKSPACE, WORKSPACE_EXCLUDED_DIRS)
        print(f"Workspace restored from: {DRIVE_WORKSPACE}")
    return state_source

restore()


## 4. Hermes AgentとOllamaの導入

Hermesは検証対象バージョンを固定します。OllamaはGemma 4に対応する最新版を公式インストーラーから導入し、
ローカルホストだけで待ち受けます。再実行時は既存インストールと既存サーバーを再利用します。


In [ ]:
def _package_version(distribution: str) -> str | None:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

installed_hermes = _package_version("hermes-agent")
if installed_hermes != HERMES_VERSION:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", f"hermes-agent[all]=={HERMES_VERSION}"],
        check=True,
    )

# Colab ships an unrelated package named "cli" that shadows Hermes' top-level
# cli.py. The launcher preloads Hermes' module into sys.modules["cli"] before
# importing hermes_cli.main, making the resolution deterministic.
hermes_distribution = importlib.metadata.distribution("hermes-agent")
real_cli_path = Path(hermes_distribution.locate_file("cli.py")).resolve()
if not real_cli_path.is_file():
    raise RuntimeError(f"Hermes cli.py was not found: {real_cli_path}")

HERMES_IMPORT_SHIM = Path("/content/hermes-import-shim")
HERMES_IMPORT_SHIM.mkdir(parents=True, exist_ok=True)
HERMES_LAUNCHER = HERMES_IMPORT_SHIM / "hermes_entry.py"
launcher_source = f"""import importlib.util
import sys

_REAL_CLI = {str(real_cli_path)!r}
_SPEC = importlib.util.spec_from_file_location("cli", _REAL_CLI)
if _SPEC is None or _SPEC.loader is None:
    raise ImportError(f"Could not load Hermes cli.py: {{_REAL_CLI}}")
_MODULE = importlib.util.module_from_spec(_SPEC)
sys.modules["cli"] = _MODULE
_SPEC.loader.exec_module(_MODULE)

from hermes_cli.main import main as hermes_main
raise SystemExit(hermes_main())
"""
HERMES_LAUNCHER.write_text(launcher_source, encoding="utf-8")

if shutil.which("ollama") is None:
    # The current Ollama Linux archive is zstd-compressed. Colab images do not
    # always include the zstd executable, so install it before the official installer.
    if shutil.which("zstd") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)

    installer_url = "https://ollama.com/install.sh"
    installer_path = Path("/tmp/ollama-install.sh")
    with urllib.request.urlopen(installer_url, timeout=60) as response:
        installer_path.write_bytes(response.read())
    installer_result = subprocess.run(
        ["bash", str(installer_path)],
        text=True,
        capture_output=True,
    )
    if installer_result.returncode != 0:
        raise RuntimeError(
            "Ollamaのインストールに失敗しました。\n"
            f"stdout:\n{installer_result.stdout[-4000:]}\n"
            f"stderr:\n{installer_result.stderr[-4000:]}"
        )

def _ollama_json(path: str, timeout: int = 5) -> dict:
    with urllib.request.urlopen(f"{OLLAMA_BASE_URL}{path}", timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

def _ollama_ready() -> bool:
    try:
        _ollama_json("/api/tags")
        return True
    except (OSError, urllib.error.URLError, json.JSONDecodeError):
        return False

OLLAMA_PROCESS = globals().get("OLLAMA_PROCESS")
if not _ollama_ready():
    ollama_environment = os.environ.copy()
    log_handle = OLLAMA_LOG.open("a", encoding="utf-8")
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=ollama_environment,
        start_new_session=True,
    )
    for _ in range(60):
        if _ollama_ready():
            break
        if OLLAMA_PROCESS.poll() is not None:
            tail = OLLAMA_LOG.read_text(encoding="utf-8", errors="replace")[-4000:]
            raise RuntimeError(f"Ollamaの起動に失敗しました。\n{tail}")
        time.sleep(1)
    else:
        raise TimeoutError(f"Ollamaが60秒以内に起動しませんでした。ログ: {OLLAMA_LOG}")

CONFIG_YAML = f"""model:
  default: "{MODEL}"
  provider: "custom"
  base_url: "{OLLAMA_BASE_URL}/v1"
  context_length: {OLLAMA_CONTEXT_LENGTH}

terminal:
  cwd: "{LOCAL_WORKSPACE}"

approvals:
  mode: "manual"
"""

if not CONFIG_FILE.exists():
    CONFIG_FILE.write_text(CONFIG_YAML, encoding="utf-8")

seed_files = {
    "MEMORY.md": "# MEMORY\n\nHermesが長期的に保持する記憶をここに蓄積します。\n",
    "USER.md": "# USER\n\nユーザーについて長期的に保持する情報をここに蓄積します。\n",
}
for filename, initial_content in seed_files.items():
    path = LOCAL_HOME / filename
    if not path.exists():
        path.write_text(initial_content, encoding="utf-8")

print(f"Hermes Agent: {_package_version('hermes-agent')}")
print(f"Ollama:       {_run_text(['ollama', '--version'])}")
print(f"Config:       {CONFIG_FILE}")
print("Ollama server: ready")


## 5. `gemma4:31b`の取得

初回は約20GBを取得します。モデルはDriveへ保存しないため、新しいColab VMでは再取得が必要です。
セルを中断せず、完了後にモデル一覧とHermes設定診断を確認してください。


In [ ]:
available_models = {
    item.get("name", "") for item in _ollama_json("/api/tags", timeout=15).get("models", [])
}
if MODEL not in available_models and f"{MODEL}:latest" not in available_models:
    subprocess.run(["ollama", "pull", MODEL], check=True)

subprocess.run(["ollama", "show", MODEL], check=True)
doctor = subprocess.run(
    ["hermes", "doctor"],
    cwd=LOCAL_WORKSPACE,
    env=os.environ.copy(),
    text=True,
    capture_output=True,
)
print(doctor.stdout)
if doctor.returncode != 0:
    print(doctor.stderr, file=sys.stderr)
    raise RuntimeError("hermes doctor が失敗しました。上の診断結果を確認してください。")

checkpoint()
print(f"{MODEL} is ready.")


## 6. Notebook操作API

- `ask_hermes(...)`: 会話を実行し、成功・失敗・中断にかかわらず最後に保存
- `new_session()`: 次の質問から新しい会話を開始
- `checkpoint()`: 手動保存
- `restore(force=True)`: Driveの正常な状態でローカルを置換
- `status()`: GPU・Ollama・モデル・Hermes・現在セッションを表示


In [ ]:
@dataclass(frozen=True)
class HermesResult:
    output: str
    returncode: int
    session_id: str | None

def _load_runtime_state() -> dict:
    if not RUNTIME_STATE_FILE.exists():
        return {"current_session_id": None, "last_checkpoint": None}
    try:
        return json.loads(RUNTIME_STATE_FILE.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {"current_session_id": None, "last_checkpoint": None}

def _save_runtime_state() -> None:
    RUNTIME_STATE_FILE.parent.mkdir(parents=True, exist_ok=True)
    temporary = RUNTIME_STATE_FILE.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(
            {
                "current_session_id": CURRENT_SESSION_ID,
                "last_checkpoint": datetime.now(timezone.utc).isoformat(),
            },
            ensure_ascii=False,
            indent=2,
        ) + "\n",
        encoding="utf-8",
    )
    temporary.replace(RUNTIME_STATE_FILE)

_runtime_state = _load_runtime_state()
CURRENT_SESSION_ID = _runtime_state.get("current_session_id")
ACTIVE_HERMES_PROCESS: subprocess.Popen[str] | None = None

def new_session() -> None:
    """Start a new conversation on the next ask_hermes call."""
    global CURRENT_SESSION_ID
    CURRENT_SESSION_ID = None
    _save_runtime_state()
    checkpoint()
    print("New session selected.")

def _stop_process(process: subprocess.Popen[str], grace_seconds: float = 5.0) -> None:
    if process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait(timeout=grace_seconds)
    except (ProcessLookupError, subprocess.TimeoutExpired):
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass

def ask_hermes(
    prompt: str,
    session_id: str | None = None,
    allow_dangerous: bool = False,
) -> HermesResult:
    """Run one Hermes turn and checkpoint state/workspace in a finally block."""
    global ACTIVE_HERMES_PROCESS, CURRENT_SESSION_ID
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("promptには空でない文字列を指定してください。")
    if not _ollama_ready():
        raise RuntimeError("Ollamaが起動していません。セットアップセルを再実行してください。")

    selected_session = session_id or CURRENT_SESSION_ID
    command = [sys.executable, str(HERMES_LAUNCHER), "chat", "-Q", "-q", prompt]
    if selected_session:
        command.extend(["--resume", selected_session])
    if allow_dangerous:
        command.append("--yolo")

    output_lines: list[str] = []
    discovered_session: str | None = selected_session
    returncode = 1
    try:
        ACTIVE_HERMES_PROCESS = subprocess.Popen(
            command,
            cwd=LOCAL_WORKSPACE,
            env=os.environ.copy(),
            stdin=subprocess.DEVNULL,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            start_new_session=True,
        )
        assert ACTIVE_HERMES_PROCESS.stdout is not None
        for line in ACTIVE_HERMES_PROCESS.stdout:
            print(line, end="")
            output_lines.append(line)
            match = re.search(r"session_id:\s*([^\s]+)", line)
            if match:
                discovered_session = match.group(1)
        returncode = ACTIVE_HERMES_PROCESS.wait()
        if discovered_session:
            CURRENT_SESSION_ID = discovered_session
        _save_runtime_state()
        if returncode != 0:
            raise RuntimeError(f"Hermesが終了コード{returncode}で失敗しました。")
        return HermesResult("".join(output_lines), returncode, discovered_session)
    except KeyboardInterrupt:
        if ACTIVE_HERMES_PROCESS is not None:
            _stop_process(ACTIVE_HERMES_PROCESS)
        print("\nHermesを中断しました。保存可能な状態をチェックポイントします。")
        raise
    finally:
        ACTIVE_HERMES_PROCESS = None
        _save_runtime_state()
        checkpoint()

def status() -> dict:
    """Print and return a compact runtime status report."""
    runtime = _load_runtime_state()
    try:
        models = [item.get("name") for item in _ollama_json("/api/tags", timeout=5).get("models", [])]
    except Exception:
        models = []
    report = {
        "gpu": gpu_name,
        "gpu_vram_gib": round(gpu_memory_mib / 1024, 1),
        "ram_gib": round(ram_gib, 1),
        "hermes_version": _package_version("hermes-agent"),
        "ollama_ready": _ollama_ready(),
        "model_present": any(name and name.startswith(MODEL) for name in models),
        "current_session_id": CURRENT_SESSION_ID,
        "last_checkpoint": runtime.get("last_checkpoint"),
        "drive_root": str(DRIVE_ROOT),
    }
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return report

def _cleanup_active_process() -> None:
    if ACTIVE_HERMES_PROCESS is not None:
        _stop_process(ACTIVE_HERMES_PROCESS)

atexit.register(_cleanup_active_process)

import shlex
from IPython.core.error import UsageError
from IPython.core.magic import register_line_cell_magic

@register_line_cell_magic
def llm(line: str, cell: str | None = None) -> None:
    """Run Hermes from %llm or %%llm and store the result in LAST_HERMES_RESULT."""
    tokens = shlex.split(line)
    allow_dangerous = False
    start_new_session = False

    while tokens and tokens[0] in {"--dangerous", "--new"}:
        option = tokens.pop(0)
        if option == "--dangerous":
            allow_dangerous = True
        elif option == "--new":
            start_new_session = True

    if tokens and tokens[0].startswith("--"):
        raise UsageError(f"Unknown %llm option: {tokens[0]}")
    if cell is not None and tokens:
        raise UsageError("With %%llm, put only --new/--dangerous on the magic line.")

    prompt = cell.strip() if cell is not None else " ".join(tokens).strip()
    if not prompt:
        raise UsageError("Usage: %llm instruction  or  %%llm followed by the instruction body")

    if start_new_session:
        new_session()

    result = ask_hermes(prompt, allow_dangerous=allow_dangerous)
    get_ipython().user_ns["LAST_HERMES_RESULT"] = result
    return None

status()
print("Magic ready: %llm instruction / %%llm + multiline instruction")


## 7. GUIを選択して起動

同じHermes状態を使い、次の2モードから選べます。

- `colab-dashboard`: Colabの認証付きプロキシ経由でWeb Dashboardを開く。追加設定は不要。
- `desktop-app`: 手元PCのHermes Desktop Appから、Tailscale内のColab Dashboardへ接続する。

Desktop Appモードでは、先にColabの「シークレット」に次を登録し、Notebookからのアクセスを許可してください。

- `TAILSCALE_AUTHKEY`: Tailscaleの再利用可能な認証キー（ephemeral推奨）
- `HERMES_DASHBOARD_USERNAME`: Dashboardのログイン名
- `HERMES_DASHBOARD_PASSWORD`: 強いログインパスワード
- `HERMES_DASHBOARD_SECRET`: 32バイト以上の固定署名シークレット

手元PCも同じTailscaleネットワークへ参加させます。起動後に表示されるURLをDesktop Appの
**Settings → Gateway → Remote gateway**へ入力し、上記ユーザー名・パスワードでサインインしてください。

停止する場合は `stop_gui()`、状態確認は `gui_status()` を使います。GUI起動中は変更検出後15秒以内を目安に自動チェックポイントし、`stop_gui()`でも最終保存します。ランタイム破棄前には念のため `checkpoint()` の完了も確認してください。

In [ ]:
import threading

GUI_PORT = 9119
GUI_AUTOSAVE_INTERVAL = 15
DASHBOARD_LOG = Path("/content/hermes-dashboard.log")
TAILSCALE_LOG = Path("/content/tailscaled.log")
TAILSCALE_SOCKET = Path("/tmp/tailscaled-hermes.sock")
TAILSCALE_STATE = Path("/tmp/tailscaled-hermes.state")
TAILSCALE_AUTH_FILE = Path("/tmp/tailscale-authkey")

DASHBOARD_PROCESS = globals().get("DASHBOARD_PROCESS")
TAILSCALED_PROCESS = globals().get("TAILSCALED_PROCESS")
CURRENT_GUI_MODE = globals().get("CURRENT_GUI_MODE")
CURRENT_GUI_URL = globals().get("CURRENT_GUI_URL")
GUI_AUTOSAVE_STOP = globals().get("GUI_AUTOSAVE_STOP")
GUI_AUTOSAVE_THREAD = globals().get("GUI_AUTOSAVE_THREAD")
CHECKPOINT_LOCK = globals().get("CHECKPOINT_LOCK") or threading.RLock()
if "_BASE_CHECKPOINT" not in globals():
    _BASE_CHECKPOINT = checkpoint

def checkpoint() -> Path:
    """Serialize checkpoint writes from notebook and GUI autosave threads."""
    with CHECKPOINT_LOCK:
        return _BASE_CHECKPOINT()

def _process_alive(process: subprocess.Popen | None) -> bool:
    return process is not None and process.poll() is None

def _log_tail(path: Path, limit: int = 5000) -> str:
    if not path.exists():
        return "(log file not created)"
    return path.read_text(encoding="utf-8", errors="replace")[-limit:]

def _colab_secret(name: str) -> str:
    from google.colab import userdata
    try:
        value = userdata.get(name)
    except Exception as exc:
        raise RuntimeError(
            f"Colab Secretsに{name}を登録し、Notebookからのアクセスを許可してください。"
        ) from exc
    if not value:
        raise RuntimeError(f"Colab Secret {name} が空です。")
    return value

def _dashboard_status(port: int = GUI_PORT) -> dict:
    with urllib.request.urlopen(f"http://127.0.0.1:{port}/api/status", timeout=5) as response:
        return json.loads(response.read().decode("utf-8"))

def _wait_for_dashboard(process: subprocess.Popen, port: int, timeout_seconds: int = 180) -> dict:
    deadline = time.time() + timeout_seconds
    last_error = "not started"
    while time.time() < deadline:
        if process.poll() is not None:
            raise RuntimeError(
                f"Hermes Dashboardが終了コード{process.returncode}で停止しました。\n"
                f"{_log_tail(DASHBOARD_LOG)}"
            )
        try:
            return _dashboard_status(port)
        except Exception as exc:
            last_error = str(exc)
            time.sleep(1)
    raise TimeoutError(
        f"Hermes Dashboardが{timeout_seconds}秒以内に起動しませんでした: {last_error}\n"
        f"{_log_tail(DASHBOARD_LOG)}"
    )

def _ensure_artifacts_dashboard_plugin() -> Path:
    """Install the local dashboard plugin that previews workspace artifacts."""
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    plugin_root = LOCAL_HOME / "plugins" / "hermes-artifacts" / "dashboard"
    dist_dir = plugin_root / "dist"
    dist_dir.mkdir(parents=True, exist_ok=True)

    manifest = {
        "name": "hermes-artifacts",
        "label": "Artifacts",
        "description": "Preview HTML, image, and data artifacts generated in the Hermes workspace.",
        "icon": "BarChart3",
        "version": "1.0.0",
        "tab": {"path": "/artifacts", "position": "after:sessions"},
        "entry": "dist/index.js",
        "css": "dist/style.css",
        "api": "plugin_api.py",
    }
    (plugin_root / "manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

    (dist_dir / "index.js").write_text(r"""
(function () {
  "use strict";

  const SDK = window.__HERMES_PLUGIN_SDK__;
  if (!SDK || !window.__HERMES_PLUGINS__) return;

  const React = SDK.React;
  const e = React.createElement;
  const apiBase = "/api/plugins/hermes-artifacts";
  const imageKinds = new Set(["image"]);
  const textKinds = new Set(["text", "data"]);

  function encodePath(path) {
    return String(path).split("/").map(encodeURIComponent).join("/");
  }

  function formatSize(bytes) {
    if (bytes < 1024) return `${bytes} B`;
    if (bytes < 1024 * 1024) return `${(bytes / 1024).toFixed(1)} KB`;
    return `${(bytes / 1024 / 1024).toFixed(1)} MB`;
  }

  async function authFetch(url, options) {
    const opts = Object.assign({}, options || {});
    const headers = new Headers(opts.headers || {});
    if (window.__HERMES_SESSION_TOKEN__) {
      headers.set("X-Hermes-Session-Token", window.__HERMES_SESSION_TOKEN__);
    }
    const response = await fetch(url, Object.assign(opts, {
      headers,
      credentials: "include",
      cache: "no-store",
    }));
    if (!response.ok) {
      let body = "";
      try { body = await response.text(); } catch (_) {}
      throw new Error(`${response.status}${body ? `: ${body}` : ""}`);
    }
    return response;
  }

  function ArtifactsPage() {
    const [items, setItems] = React.useState([]);
    const [selected, setSelected] = React.useState(null);
    const [textPreview, setTextPreview] = React.useState("");
    const [previewUrl, setPreviewUrl] = React.useState("");
    const [loading, setLoading] = React.useState(true);
    const [error, setError] = React.useState("");

    const loadItems = React.useCallback(async function () {
      setLoading(true);
      setError("");
      try {
        const data = await SDK.fetchJSON(`${apiBase}/artifacts`);
        setItems(data.artifacts || []);
        setSelected(function (current) {
          if (current && (data.artifacts || []).some((item) => item.path === current.path)) return current;
          return (data.artifacts || [])[0] || null;
        });
      } catch (err) {
        setError(err instanceof Error ? err.message : String(err));
      } finally {
        setLoading(false);
      }
    }, []);

    React.useEffect(function () {
      loadItems();
    }, [loadItems]);

    React.useEffect(function () {
      let cancelled = false;
      async function loadText() {
        setTextPreview("");
        if (!selected || !textKinds.has(selected.kind)) return;
        try {
          const response = await authFetch(`${apiBase}/text/${encodePath(selected.path)}`);
          const body = await response.text();
          if (!cancelled) setTextPreview(body);
        } catch (err) {
          if (!cancelled) setTextPreview(err instanceof Error ? err.message : String(err));
        }
      }
      loadText();
      return function () { cancelled = true; };
    }, [selected]);

    React.useEffect(function () {
      let cancelled = false;
      let objectUrl = "";
      async function loadBlob() {
        setPreviewUrl("");
        if (!selected || textKinds.has(selected.kind)) return;
        try {
          const response = await authFetch(`${apiBase}/file/${encodePath(selected.path)}`);
          const blob = await response.blob();
          objectUrl = URL.createObjectURL(blob);
          if (cancelled) {
            URL.revokeObjectURL(objectUrl);
          } else {
            setPreviewUrl(objectUrl);
          }
        } catch (err) {
          if (!cancelled) setError(err instanceof Error ? err.message : String(err));
        }
      }
      loadBlob();
      return function () {
        cancelled = true;
        if (objectUrl) URL.revokeObjectURL(objectUrl);
      };
    }, [selected]);

    function renderPreview() {
      if (!selected) {
        return e("div", { className: "hermes-artifacts-empty" }, "No artifacts yet.");
      }
      if (selected.kind === "html") {
        if (!previewUrl) return e("div", { className: "hermes-artifacts-empty" }, "Loading preview...");
        return e("iframe", {
          key: selected.path,
          className: "hermes-artifacts-frame",
          src: previewUrl,
          sandbox: "allow-scripts allow-forms allow-popups allow-downloads",
          title: selected.name,
        });
      }
      if (imageKinds.has(selected.kind)) {
        if (!previewUrl) return e("div", { className: "hermes-artifacts-empty" }, "Loading preview...");
        return e("div", { className: "hermes-artifacts-image-wrap" },
          e("img", { src: previewUrl, alt: selected.name, className: "hermes-artifacts-image" })
        );
      }
      if (textKinds.has(selected.kind)) {
        return e("pre", { className: "hermes-artifacts-text" }, textPreview || "Loading preview...");
      }
      return e("div", { className: "hermes-artifacts-empty" },
        previewUrl
          ? e("a", { href: previewUrl, download: selected.name }, "Download artifact")
          : "Loading artifact..."
      );
    }

    return e("div", { className: "hermes-artifacts-page" },
      e("div", { className: "hermes-artifacts-header" },
        e("div", null,
          e("h1", null, "Artifacts"),
          e("p", null, "Files saved under /content/hermes-workspace/artifacts")
        ),
        e("button", { type: "button", onClick: loadItems, disabled: loading }, loading ? "Loading" : "Refresh")
      ),
      error ? e("div", { className: "hermes-artifacts-error" }, error) : null,
      e("div", { className: "hermes-artifacts-layout" },
        e("aside", { className: "hermes-artifacts-list" },
          items.length === 0 && !loading ? e("div", { className: "hermes-artifacts-empty" }, "No artifacts found.") : null,
          items.map(function (item) {
            const active = selected && selected.path === item.path;
            return e("button", {
              key: item.path,
              type: "button",
              className: active ? "hermes-artifacts-item is-active" : "hermes-artifacts-item",
              onClick: function () { setSelected(item); },
            },
              e("span", { className: "hermes-artifacts-name" }, item.name),
              e("span", { className: "hermes-artifacts-meta" }, `${item.kind.toUpperCase()} - ${formatSize(item.size)}`),
              e("span", { className: "hermes-artifacts-meta" }, new Date(item.mtime * 1000).toLocaleString())
            );
          })
        ),
        e("main", { className: "hermes-artifacts-preview" }, renderPreview())
      )
    );
  }

  window.__HERMES_PLUGINS__.register("hermes-artifacts", ArtifactsPage);
})();
""".lstrip(), encoding="utf-8")

    (dist_dir / "style.css").write_text(r"""
.hermes-artifacts-page {
  display: flex;
  min-height: calc(100vh - 8rem);
  flex-direction: column;
  gap: 1rem;
}
.hermes-artifacts-header {
  display: flex;
  align-items: flex-end;
  justify-content: space-between;
  gap: 1rem;
  border-bottom: 1px solid var(--color-border, rgba(148, 163, 184, 0.28));
  padding-bottom: 0.75rem;
}
.hermes-artifacts-header h1 {
  margin: 0;
  font-size: 1.5rem;
  line-height: 1.2;
}
.hermes-artifacts-header p,
.hermes-artifacts-meta {
  color: var(--color-muted-foreground, #8a93a3);
  font-size: 0.8125rem;
}
.hermes-artifacts-header button,
.hermes-artifacts-item {
  border: 1px solid var(--color-border, rgba(148, 163, 184, 0.28));
  background: var(--color-background, transparent);
  color: var(--color-foreground, inherit);
  cursor: pointer;
}
.hermes-artifacts-header button {
  min-height: 2rem;
  padding: 0 0.75rem;
}
.hermes-artifacts-layout {
  display: grid;
  grid-template-columns: minmax(15rem, 22rem) minmax(0, 1fr);
  min-height: 0;
  flex: 1;
  gap: 1rem;
}
.hermes-artifacts-list {
  display: flex;
  min-height: 0;
  flex-direction: column;
  gap: 0.5rem;
  overflow: auto;
}
.hermes-artifacts-item {
  display: flex;
  flex-direction: column;
  gap: 0.25rem;
  padding: 0.75rem;
  text-align: left;
}
.hermes-artifacts-item:hover,
.hermes-artifacts-item.is-active {
  border-color: var(--color-ring, #5eead4);
  background: color-mix(in srgb, var(--color-primary, #14b8a6) 12%, transparent);
}
.hermes-artifacts-name {
  overflow-wrap: anywhere;
  font-weight: 600;
}
.hermes-artifacts-preview {
  min-width: 0;
  min-height: 32rem;
  overflow: hidden;
  border: 1px solid var(--color-border, rgba(148, 163, 184, 0.28));
  background: var(--color-card, rgba(15, 23, 42, 0.2));
}
.hermes-artifacts-frame,
.hermes-artifacts-text {
  width: 100%;
  height: 100%;
  min-height: 32rem;
  border: 0;
  background: white;
}
.hermes-artifacts-text {
  box-sizing: border-box;
  margin: 0;
  overflow: auto;
  padding: 1rem;
  background: var(--color-background, #0b1020);
  color: var(--color-foreground, #e5e7eb);
  white-space: pre-wrap;
}
.hermes-artifacts-image-wrap {
  display: flex;
  height: 100%;
  min-height: 32rem;
  align-items: center;
  justify-content: center;
  overflow: auto;
  padding: 1rem;
}
.hermes-artifacts-image {
  max-width: 100%;
  max-height: 80vh;
}
.hermes-artifacts-empty,
.hermes-artifacts-error {
  padding: 1rem;
  color: var(--color-muted-foreground, #8a93a3);
}
.hermes-artifacts-error {
  border: 1px solid var(--color-destructive, #ef4444);
  color: var(--color-destructive, #ef4444);
}
@media (max-width: 900px) {
  .hermes-artifacts-layout {
    grid-template-columns: 1fr;
  }
  .hermes-artifacts-preview,
  .hermes-artifacts-frame,
  .hermes-artifacts-text,
  .hermes-artifacts-image-wrap {
    min-height: 26rem;
  }
}
""".lstrip(), encoding="utf-8")

    (plugin_root / "plugin_api.py").write_text(r"""
from __future__ import annotations

import mimetypes
import os
from pathlib import Path

from fastapi import APIRouter, HTTPException
from fastapi.responses import FileResponse, PlainTextResponse

router = APIRouter()
ARTIFACTS_ROOT = Path(os.environ.get("HERMES_ARTIFACTS_DIR", "/content/hermes-workspace/artifacts")).resolve()
TEXT_EXTS = {".csv", ".json", ".txt", ".md", ".tsv", ".yaml", ".yml", ".xml"}
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".gif", ".webp", ".bmp", ".tiff", ".svg"}
HTML_EXTS = {".html", ".htm"}
MAX_TEXT_BYTES = 1_000_000


def _resolve_artifact(relative_path: str) -> Path:
    ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
    target = (ARTIFACTS_ROOT / relative_path).resolve()
    try:
        target.relative_to(ARTIFACTS_ROOT)
    except ValueError as exc:
        raise HTTPException(status_code=403, detail="Path traversal blocked") from exc
    if not target.is_file():
        raise HTTPException(status_code=404, detail="Artifact not found")
    return target


def _kind(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in HTML_EXTS:
        return "html"
    if suffix in IMAGE_EXTS:
        return "image"
    if suffix in TEXT_EXTS:
        return "data" if suffix in {".csv", ".tsv", ".json", ".yaml", ".yml", ".xml"} else "text"
    return "file"


@router.get("/artifacts")
async def list_artifacts():
    ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
    artifacts = []
    for path in ARTIFACTS_ROOT.rglob("*"):
        if not path.is_file() or path.is_symlink():
            continue
        relative = path.relative_to(ARTIFACTS_ROOT).as_posix()
        stat = path.stat()
        artifacts.append({
            "path": relative,
            "name": path.name,
            "size": stat.st_size,
            "mtime": stat.st_mtime,
            "kind": _kind(path),
            "extension": path.suffix.lower(),
        })
    artifacts.sort(key=lambda item: item["mtime"], reverse=True)
    return {"root": str(ARTIFACTS_ROOT), "artifacts": artifacts}


@router.get("/file/{relative_path:path}")
async def get_artifact(relative_path: str):
    target = _resolve_artifact(relative_path)
    media_type = mimetypes.guess_type(target.name)[0] or "application/octet-stream"
    return FileResponse(target, media_type=media_type)


@router.get("/text/{relative_path:path}")
async def get_text_artifact(relative_path: str):
    target = _resolve_artifact(relative_path)
    if target.suffix.lower() not in TEXT_EXTS:
        raise HTTPException(status_code=415, detail="Artifact is not text-previewable")
    data = target.read_bytes()[:MAX_TEXT_BYTES]
    text = data.decode("utf-8", errors="replace")
    if target.stat().st_size > MAX_TEXT_BYTES:
        text += "\n\n[Preview truncated at 1 MB]"
    return PlainTextResponse(text)
""".lstrip(), encoding="utf-8")

    print(f"Artifacts plugin ready: {plugin_root}")
    print(f"Artifacts directory: {ARTIFACTS_DIR}")
    return plugin_root

def _start_dashboard(
    host: str,
    port: int,
    auth: dict[str, str] | None = None,
    insecure: bool = False,
) -> dict:
    global DASHBOARD_PROCESS
    _ensure_artifacts_dashboard_plugin()
    environment = os.environ.copy()
    environment["HERMES_ARTIFACTS_DIR"] = str(ARTIFACTS_DIR)
    for name in (
        "HERMES_DASHBOARD_BASIC_AUTH_USERNAME",
        "HERMES_DASHBOARD_BASIC_AUTH_PASSWORD",
        "HERMES_DASHBOARD_BASIC_AUTH_PASSWORD_HASH",
        "HERMES_DASHBOARD_BASIC_AUTH_SECRET",
    ):
        environment.pop(name, None)
    if auth:
        environment.update(auth)

    command = [
        sys.executable,
        str(HERMES_LAUNCHER),
        "dashboard",
        "--no-open",
        "--host",
        host,
        "--port",
        str(port),
    ]
    if insecure:
        command.append("--insecure")

    log_handle = DASHBOARD_LOG.open("a", encoding="utf-8")
    try:
        DASHBOARD_PROCESS = subprocess.Popen(
            command,
            cwd=LOCAL_WORKSPACE,
            env=environment,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            text=True,
            start_new_session=True,
        )
    finally:
        log_handle.close()
    return _wait_for_dashboard(DASHBOARD_PROCESS, port)

def _ensure_tailscale() -> None:
    if shutil.which("tailscale") and shutil.which("tailscaled"):
        return
    installer = Path("/tmp/tailscale-install.sh")
    with urllib.request.urlopen("https://tailscale.com/install.sh", timeout=60) as response:
        installer.write_bytes(response.read())
    result = subprocess.run(
        ["bash", str(installer)],
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            "Tailscaleのインストールに失敗しました。\n"
            f"stdout:\n{result.stdout[-3000:]}\n"
            f"stderr:\n{result.stderr[-3000:]}"
        )

def _tailscale_command(*args: str, check: bool = True, timeout: int = 120) -> subprocess.CompletedProcess[str]:
    command = ["tailscale", f"--socket={TAILSCALE_SOCKET}", *args]
    return subprocess.run(
        command,
        text=True,
        capture_output=True,
        check=check,
        timeout=timeout,
    )

def _start_tailscale(port: int, auth_key: str) -> str:
    global TAILSCALED_PROCESS
    _ensure_tailscale()

    if not _process_alive(TAILSCALED_PROCESS):
        TAILSCALE_SOCKET.unlink(missing_ok=True)
        log_handle = TAILSCALE_LOG.open("a", encoding="utf-8")
        try:
            TAILSCALED_PROCESS = subprocess.Popen(
                [
                    "tailscaled",
                    "--tun=userspace-networking",
                    f"--state={TAILSCALE_STATE}",
                    f"--socket={TAILSCALE_SOCKET}",
                ],
                stdout=log_handle,
                stderr=subprocess.STDOUT,
                text=True,
                start_new_session=True,
            )
        finally:
            log_handle.close()

    deadline = time.time() + 30
    while time.time() < deadline and not TAILSCALE_SOCKET.exists():
        if TAILSCALED_PROCESS.poll() is not None:
            raise RuntimeError(f"tailscaledの起動に失敗しました。\n{_log_tail(TAILSCALE_LOG)}")
        time.sleep(0.5)
    if not TAILSCALE_SOCKET.exists():
        raise TimeoutError(f"tailscaledのsocketが作成されませんでした。\n{_log_tail(TAILSCALE_LOG)}")

    TAILSCALE_AUTH_FILE.write_text(auth_key.strip() + "\n", encoding="utf-8")
    TAILSCALE_AUTH_FILE.chmod(0o600)
    try:
        up = _tailscale_command(
            "up",
            f"--auth-key=file:{TAILSCALE_AUTH_FILE}",
            "--hostname=hermes-colab",
            "--accept-dns=false",
            check=False,
        )
    finally:
        TAILSCALE_AUTH_FILE.unlink(missing_ok=True)
    if up.returncode != 0:
        raise RuntimeError(
            "Tailscaleへの接続に失敗しました。認証キーとtailnet設定を確認してください。\n"
            f"{up.stdout}\n{up.stderr}"
        )

    _tailscale_command("serve", "reset", check=False, timeout=20)
    serve_mode = "https"
    serve_errors = []
    try:
        served = _tailscale_command("serve", "--yes", "--bg", str(port), check=False, timeout=30)
    except subprocess.TimeoutExpired as exc:
        serve_errors.append(f"HTTPS Serve timed out: {exc}")
        served = None
    if served is None or served.returncode != 0:
        if served is not None:
            serve_errors.append(f"HTTPS Serve failed:\n{served.stdout}\n{served.stderr}")
        # Colab often cannot complete Tailscale's HTTPS certificate consent flow.
        # Fall back to tailnet-only HTTP; Tailscale still encrypts transport.
        serve_mode = "http"
        fallback = _tailscale_command("serve", "--yes", "--bg", "--http=80", str(port), check=False, timeout=30)
        if fallback.returncode != 0:
            raise RuntimeError(
                "Tailscale Serveの開始に失敗しました。\n"
                + "\n".join(serve_errors)
                + f"\nHTTP fallback failed:\n{fallback.stdout}\n{fallback.stderr}"
            )

    status_data = json.loads(_tailscale_command("status", "--json").stdout)
    dns_name = str(status_data.get("Self", {}).get("DNSName", "")).rstrip(".")
    if not dns_name:
        raise RuntimeError("TailscaleのDNS名を取得できませんでした。")
    return f"{serve_mode}://{dns_name}"

def _colab_dashboard_url(port: int) -> str:
    from google.colab.output import eval_js
    from IPython.display import HTML, display

    url = str(eval_js(f"google.colab.kernel.proxyPort({port})"))
    display(HTML(
        f'<p><a href="{url}" target="_blank" rel="noopener noreferrer">'
        "Hermes Web Dashboardを新しいタブで開く</a></p>"
    ))
    return url

def _gui_state_signature() -> tuple:
    entries = []
    for root, excluded_dirs, excluded_files in (
        (LOCAL_HOME, HOME_EXCLUDED_DIRS, HOME_EXCLUDED_FILES),
        (LOCAL_WORKSPACE, WORKSPACE_EXCLUDED_DIRS, set()),
    ):
        if not root.exists():
            continue
        for path in root.rglob("*"):
            relative = path.relative_to(root)
            if path.is_symlink() or any(part in excluded_dirs for part in relative.parts):
                continue
            if relative.name in excluded_files or not path.is_file():
                continue
            try:
                stat = path.stat()
            except OSError:
                continue
            entries.append((str(root), str(relative), stat.st_size, stat.st_mtime_ns))
    return tuple(sorted(entries))

def _stop_gui_autosave(final_checkpoint: bool = False) -> None:
    global GUI_AUTOSAVE_STOP, GUI_AUTOSAVE_THREAD
    stop_event = GUI_AUTOSAVE_STOP
    thread = GUI_AUTOSAVE_THREAD
    if stop_event is not None:
        stop_event.set()
    if thread is not None and thread.is_alive() and thread is not threading.current_thread():
        thread.join(timeout=20)
    GUI_AUTOSAVE_STOP = None
    GUI_AUTOSAVE_THREAD = None
    if final_checkpoint:
        checkpoint()

def _start_gui_autosave() -> None:
    global GUI_AUTOSAVE_STOP, GUI_AUTOSAVE_THREAD
    if GUI_AUTOSAVE_THREAD is not None and GUI_AUTOSAVE_THREAD.is_alive():
        return
    checkpoint()
    stop_event = threading.Event()
    GUI_AUTOSAVE_STOP = stop_event
    initial_signature = _gui_state_signature()

    def autosave_worker() -> None:
        last_signature = initial_signature
        while not stop_event.wait(GUI_AUTOSAVE_INTERVAL):
            current_signature = _gui_state_signature()
            if current_signature == last_signature:
                continue
            try:
                checkpoint()
                last_signature = _gui_state_signature()
                print("GUI autosave checkpoint completed.")
            except Exception as exc:
                print(f"GUI autosave failed: {exc}")

    GUI_AUTOSAVE_THREAD = threading.Thread(
        target=autosave_worker,
        name="hermes-gui-autosave",
        daemon=True,
    )
    GUI_AUTOSAVE_THREAD.start()

def stop_gui() -> None:
    global DASHBOARD_PROCESS, TAILSCALED_PROCESS, CURRENT_GUI_MODE, CURRENT_GUI_URL
    had_running_gui = _process_alive(DASHBOARD_PROCESS) or _process_alive(TAILSCALED_PROCESS)
    if TAILSCALE_SOCKET.exists() and shutil.which("tailscale"):
        try:
            _tailscale_command("serve", "reset", check=False, timeout=20)
        except Exception:
            pass
    if _process_alive(DASHBOARD_PROCESS):
        _stop_process(DASHBOARD_PROCESS)
    if _process_alive(TAILSCALED_PROCESS):
        _stop_process(TAILSCALED_PROCESS)
    DASHBOARD_PROCESS = None
    TAILSCALED_PROCESS = None
    CURRENT_GUI_MODE = None
    CURRENT_GUI_URL = None
    _stop_gui_autosave(final_checkpoint=had_running_gui)
    print("Hermes GUI processes stopped.")

def gui_status() -> dict:
    api_status = {}
    if _process_alive(DASHBOARD_PROCESS):
        try:
            api_status = _dashboard_status(GUI_PORT)
        except Exception as exc:
            api_status = {"error": str(exc)}
    report = {
        "mode": CURRENT_GUI_MODE,
        "url": CURRENT_GUI_URL,
        "dashboard_running": _process_alive(DASHBOARD_PROCESS),
        "tailscale_running": _process_alive(TAILSCALED_PROCESS),
        "autosave_running": GUI_AUTOSAVE_THREAD is not None and GUI_AUTOSAVE_THREAD.is_alive(),
        "auth_required": api_status.get("auth_required"),
        "auth_providers": api_status.get("auth_providers", []),
        "artifacts_dir": str(ARTIFACTS_DIR),
    }
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return report

def start_gui(mode: str = "colab-dashboard", port: int = GUI_PORT) -> dict:
    global CURRENT_GUI_MODE, CURRENT_GUI_URL
    if mode not in {"colab-dashboard", "desktop-app"}:
        raise ValueError("modeは 'colab-dashboard' または 'desktop-app' を指定してください。")

    stop_gui()
    try:
        if mode == "colab-dashboard":
            # Colab proxy uses its own Host header. Bind all VM interfaces so
            # Hermes accepts that header; access is still restricted by Colab's
            # authenticated proxy, and the VM port is not directly published.
            dashboard = _start_dashboard("0.0.0.0", port, insecure=True)
            if dashboard.get("auth_required"):
                raise RuntimeError("loopback Dashboardで予期せず認証が有効になっています。")
            url = _colab_dashboard_url(port)
        else:
            auth_key = _colab_secret("TAILSCALE_AUTHKEY")
            username = _colab_secret("HERMES_DASHBOARD_USERNAME")
            password = _colab_secret("HERMES_DASHBOARD_PASSWORD")
            signing_secret = _colab_secret("HERMES_DASHBOARD_SECRET")
            if len(signing_secret) < 32:
                raise RuntimeError("HERMES_DASHBOARD_SECRETは32文字以上にしてください。")
            dashboard = _start_dashboard(
                "0.0.0.0",
                port,
                {
                    "HERMES_DASHBOARD_BASIC_AUTH_USERNAME": username,
                    "HERMES_DASHBOARD_BASIC_AUTH_PASSWORD": password,
                    "HERMES_DASHBOARD_BASIC_AUTH_SECRET": signing_secret,
                },
            )
            if not dashboard.get("auth_required") or "basic" not in dashboard.get("auth_providers", []):
                raise RuntimeError("DashboardのBasic認証が有効になっていません。")
            url = _start_tailscale(port, auth_key)
            print("Desktop Appの Settings → Gateway → Remote gateway に次を設定してください:")
            print(url)
            print(f"Username: {username}")

        CURRENT_GUI_MODE = mode
        CURRENT_GUI_URL = url
        _start_gui_autosave()
        return gui_status()
    except Exception:
        stop_gui()
        raise

def _cleanup_gui_processes() -> None:
    if _process_alive(DASHBOARD_PROCESS) or _process_alive(TAILSCALED_PROCESS):
        stop_gui()

atexit.register(_cleanup_gui_processes)
print("GUI ready: start_gui('colab-dashboard') or start_gui('desktop-app')")
print(f"Artifacts tab will show files from: {ARTIFACTS_DIR}")


In [ ]:
GUI_MODE = "colab-dashboard"  # @param ["colab-dashboard", "desktop-app"]
GUI_INFO = start_gui(GUI_MODE)

## 8. 動作確認

最初の呼び出しで新しいセッションを作り、セッションID・SQLite・workspaceをDriveへ保存します。
危険操作は許可しません。


In [ ]:
result = ask_hermes(
    "日本語で一文だけ自己紹介し、現在の作業ディレクトリ名を答えてください。",
    allow_dangerous=False,
)
print(f"\nSaved session: {result.session_id}")


## 9. 継続利用

同じセッションを自動継続するため、通常は次のmagicだけを使います。

1行の指示:

```text
%llm 日本語で今日の作業方針を提案してください。
```

複数行の指示は、セル先頭を **`%%llm`** にします（`%`が2個です）。

```text
%%llm
workspaceの内容を確認してください。
その後、次に行うべき作業を3件提案してください。
```

オプション:

- `--new`: 新規セッションで開始
- `--dangerous`: その呼び出しだけ危険操作を許可
- 最後の戻り値は `LAST_HERMES_RESULT` に保存


In [ ]:
%%llm
現在のセッションが継続されていることを、日本語で一文だけ説明してください。


## 演習・運用上の注意

**演習:** `MyDrive/Hermes/state/SOUL.md` またはローカルの
`/content/hermes-home/SOUL.md` を編集し、`checkpoint()` 後に新しいランタイムから復元できることを確認してください。

**よくある問題:**

- T4/L4等が割り当てられた場合はハードウェア検査で停止します。A100へ変更してください。
- ランタイム切断中のターンは保存されない場合があります。重要な変更後は `checkpoint()` の完了表示を確認してください。
- `.env`、ログ、キャッシュ、Ollamaモデルは意図的にDrive同期対象外です。将来APIキーを使う場合はColab Secretsを使ってください。
- Driveの状態へ戻す場合は、実行中のHermesがないことを確認して `restore(force=True)` を呼びます。
